In [8]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [9]:
q1 = "Can I still join the course after the start date?"


In [10]:
v1 = model.encode(q1)

In [11]:
v1.shape

(384,)

In [12]:
v1

array([ 2.13903598e-02, -7.39799812e-02,  1.42071780e-03,  2.13816296e-02,
        2.45113764e-02,  3.15582640e-02, -1.10839747e-01, -1.05017491e-01,
       -6.18259348e-02, -6.42313063e-03,  3.72394780e-03,  9.06393304e-02,
       -9.49935894e-03,  6.53976873e-02,  1.10946447e-02, -2.10097339e-02,
       -3.35125588e-02, -4.31677438e-02,  9.96348914e-03,  1.41969407e-02,
       -6.40415177e-02, -7.04180077e-03, -7.91188106e-02,  5.80030568e-02,
        1.30212447e-03,  4.19732416e-03,  5.70978969e-02,  6.39447495e-02,
        2.49903128e-02, -3.95876579e-02, -3.79505977e-02,  2.70394646e-02,
        1.79423336e-02,  1.72272027e-02,  3.43311541e-02,  9.29057132e-03,
        5.86054437e-02, -4.97789495e-02, -5.05369715e-03,  4.34328243e-02,
       -1.56623349e-02, -2.97534615e-02, -5.13327494e-03,  5.13414592e-02,
        6.16066577e-03,  6.86980486e-02, -1.29505685e-02, -5.61938770e-02,
       -1.08264964e-02,  5.96683957e-02,  5.29939570e-02, -3.42755467e-02,
       -4.15274352e-02, -

In [13]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [14]:
v1.dot(dv)

np.float32(0.32332408)

In [15]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [16]:
v2.dot(dv)

np.float32(0.019730505)

In [17]:
import shutil
shutil.copy("ingest.py", "02-vector-search/ingest.py")

FileNotFoundError: [Errno 2] No such file or directory: 'ingest.py'

In [18]:
import os
print(os.getcwd())

c:\Users\hamza\llm-zoomcamp-mywork\02-vector-Search


In [19]:
import shutil
shutil.copy("../ingest.py", "ingest.py")

'ingest.py'

In [20]:
from ingest import load_faq_data

documents = load_faq_data()

In [21]:
documents[10]

{'id': '316180784f',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: How many hours per week am I expected to spend on this course?',
 'answer': 'It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'}

In [22]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [23]:
texts[10]

'Course: How many hours per week am I expected to spend on this course? It depends on your background and previous experience with modules. It is expected to require about 5 - 15 hours per week.\n\nYou can also calculate it yourself using [this data](https://github.com/DataTalksClub/zoomcamp-analytics/tree/main/data/de-zoomcamp-2023) and then update this answer.'

In [25]:
len(texts)


1401

In [26]:
from tqdm.auto import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/29 [00:00<?, ?it/s]

1401

In [28]:
vectors[10].shape

(384,)

In [29]:
import numpy as np
X = np.array(vectors)

In [30]:
X.shape

(1401, 384)

In [31]:
X

array([[-0.02670618, -0.12245759,  0.01594413, ..., -0.00230652,
        -0.112184  , -0.02365559],
       [-0.01099556, -0.11074748, -0.02536941, ...,  0.09022233,
        -0.02697358,  0.0197567 ],
       [-0.08896547, -0.06128181,  0.00775604, ...,  0.0405971 ,
         0.00479279, -0.02745942],
       ...,
       [ 0.00878648, -0.07507779,  0.0273208 , ..., -0.00520804,
         0.01720908,  0.03526431],
       [-0.01129572,  0.04223462, -0.03605102, ..., -0.03297512,
        -0.00711077, -0.01410813],
       [-0.01859395, -0.00951882, -0.05152113, ...,  0.04781634,
         0.02650542,  0.07470337]], shape=(1401, 384), dtype=float32)

In [33]:
scores = X.dot(v1)

In [34]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(2), np.float32(0.7629411))

In [40]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [41]:
top5 = np.argsort(scores)[-5:]

In [42]:
top5

array([   7, 1009,  567, 1150,    2])

In [43]:
scores[top5]

array([0.56009996, 0.6536313 , 0.71921325, 0.7579373 , 0.7629411 ],
      dtype=float32)

In [48]:
top5 = np.argsort(scores)[-5:]

In [49]:
top5 = top5[::-1]
top5

array([   2, 1150,  567, 1009,    7])

In [50]:
scores[top5]

array([0.7629411 , 0.7579373 , 0.71921325, 0.6536313 , 0.56009996],
      dtype=float32)

In [51]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.7629411
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579373
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.71921325
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Relate

In [53]:
from minsearch import VectorSearch

In [54]:
vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [55]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [56]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [57]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [58]:
shutil.copy("../rag_helper.py", "rag_helper.py")

'rag_helper.py'

In [59]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [60]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [61]:
from rag_helper import RAGBase

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

In [62]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes, you can still join. If you want to receive a certificate, make sure to submit your project while submissions are still being accepted.'

In [63]:

class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [64]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
)

In [65]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes — you can still join. If you want a certificate, make sure to submit your project while submissions are still open.'

In [66]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [67]:
vs_index.fit(vectors, documents)

In [68]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)

In [69]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [70]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [71]:
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nTo get the certificate, you need to finish a capstone project and complete the\nrequired peer reviews. Homework is not required. You can work through the\nmaterial and prepare your project in self-paced mode, but project submission and\npeer review must happen while a live cohort is accepting them.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  '

In [72]:
vs_index.close()